# 01 - Data Acquisition

Downloads all raw data sources needed for the county-level swing analysis:
- MEDSL county presidential election returns (2000-2024)
- ACS 5-year demographics (2023 vintage)
- BLS state unemployment rates
- NCHS urban-rural classification codes
- Census gazetteer (county land area)

## Election returns

County-level presidential returns from the MIT Election Data + Science Lab (MEDSL). 
The dataset covers 2000-2024 and is hosted on Harvard Dataverse.

In [1]:
import sys
import os

sys.path.insert(0, os.path.join(os.getcwd(), '..'))

from src.data import (
    download_medsl,
    download_nchs,
    download_gazetteer,
    download_bls_laus,
    fetch_acs_demographics,
)

In [ ]:
# Census API key - sign up free at https://api.census.gov/data/key_signup.html

API_KEY = os.environ.get('CENSUS_API_KEY', 'API_KEY')
if not API_KEY:
    raise EnvironmentError(
        "CENSUS_API_KEY not found. "
        "Get a free key at https://api.census.gov/data/key_signup.html"
    )

In [3]:
import pandas as pd

medsl_path = download_medsl()
election_raw = pd.read_csv(medsl_path, dtype={'county_fips': str})
print(f"Shape: {election_raw.shape}")
print(f"Years: {sorted(election_raw['year'].unique())}")
election_raw.head()

Already exists: c:\Users\kaleb\OneDrive\Desktop\Portfolio\us-election-county-swing\notebooks\..\src\..\data\raw\countypres_2000-2024.csv
Shape: (94151, 12)
Years: [np.int64(2000), np.int64(2004), np.int64(2008), np.int64(2012), np.int64(2016), np.int64(2020), np.int64(2024)]


,state,county_name,year,state_po,county_fips,office,candidate,party,candidatevotes,totalvotes,version,mode
0,ALABAMA,AUTAUGA,2024,AL,1001,US PRESIDENT,OTHER,OTHER,293.0,28281,20260225,TOTAL
1,ALABAMA,AUTAUGA,2024,AL,1001,US PRESIDENT,CHASE OLIVER,LIBERTARIAN,65.0,28281,20260225,TOTAL
2,ALABAMA,AUTAUGA,2024,AL,1001,US PRESIDENT,KAMALA D HARRIS,DEMOCRAT,7439.0,28281,20260225,TOTAL
3,ALABAMA,AUTAUGA,2024,AL,1001,US PRESIDENT,DONALD J TRUMP,REPUBLICAN,20484.0,28281,20260225,TOTAL
4,ALABAMA,BALDWIN,2024,AL,1003,US PRESIDENT,OTHER,OTHER,1276.0,122249,20260225,TOTAL


## Geographic classifications

NCHS urban-rural codes (6-category scheme from 2013) and the Census 
gazetteer for county land area, used to compute population density.

In [4]:
nchs_path = download_nchs()
nchs_raw = pd.read_csv(nchs_path, encoding='latin-1')
print(f"Shape: {nchs_raw.shape}")
print(f"Columns: {list(nchs_raw.columns)}")
nchs_raw.head()

Already exists: c:\Users\kaleb\OneDrive\Desktop\Portfolio\us-election-county-swing\notebooks\..\src\..\data\raw\nchs_urban_rural.csv
Shape: (3160, 11)
Columns: ['STFIPS', 'CTYFIPS', 'ST_ABBREV', 'CTYNAME', 'CBSATITLE', 'CBSAPOP', 'CTYPOP', 'CODE2023', 'CODE2013', 'CODE2006', 'CODE1990']


,STFIPS,CTYFIPS,ST_ABBREV,CTYNAME,CBSATITLE,CBSAPOP,CTYPOP,CODE2023,CODE2013,CODE2006,CODE1990
0,1,1,AL,Autauga County,"Montgomery, AL",385460.0,59759.0,3.0,3.0,3.0,3.0
1,1,3,AL,Baldwin County,"Daphne-Fairhope-Foley, AL",246435.0,246435.0,4.0,4.0,5.0,3.0
2,1,5,AL,Barbour County,"Eufaula, AL-GA",26955.0,24706.0,5.0,6.0,5.0,5.0
3,1,7,AL,Bibb County,"Birmingham, AL",1181196.0,22005.0,2.0,2.0,2.0,6.0
4,1,9,AL,Blount County,"Birmingham, AL",1181196.0,59512.0,2.0,2.0,2.0,3.0


In [5]:
gaz_path = download_gazetteer()
gaz_raw = pd.read_csv(gaz_path, sep='\t', dtype=str)
print(f"Shape: {gaz_raw.shape}")
print(f"Columns: {list(gaz_raw.columns)}")
gaz_raw.head()

Already exists: c:\Users\kaleb\OneDrive\Desktop\Portfolio\us-election-county-swing\notebooks\..\src\..\data\raw\gazetteer_counties.txt
Shape: (3222, 10)
Columns: ['USPS', 'GEOID', 'ANSICODE', 'NAME', 'ALAND', 'AWATER', 'ALAND_SQMI', 'AWATER_SQMI', 'INTPTLAT', 'INTPTLONG                                                                                                               ']


,USPS,GEOID,ANSICODE,NAME,ALAND,AWATER,ALAND_SQMI,AWATER_SQMI,INTPTLAT,INTPTLONG
0,AL,01001,00161526,Autauga County,1539631459,25677536,594.455,9.914,32.532237,-86.64644 ...
1,AL,01003,00161527,Baldwin County,4117781416,1132830835,1589.884,437.388,30.659218,-87.746067 ...
2,AL,01005,00161528,Barbour County,2292160151,50523213,885.008,19.507,31.870253,-85.405104 ...
3,AL,01007,00161529,Bibb County,1612188713,9572302,622.47,3.696,33.015893,-87.127148 ...
4,AL,01009,00161530,Blount County,1670259099,14860281,644.891,5.738,33.977358,-86.56644 ...


## State unemployment data

State-level unemployment rates from the BLS Local Area Unemployment Statistics 
(LAUS) program, fetched via the BLS public API for 2020 and 2024.

In [6]:
bls_path = download_bls_laus()
print(f"Downloaded to {bls_path}")

Fetching state unemployment rates from BLS API...
Saved 1560 records to c:\Users\kaleb\OneDrive\Desktop\Portfolio\us-election-county-swing\notebooks\..\src\..\data\raw\bls_laus_states.json
Downloaded to c:\Users\kaleb\OneDrive\Desktop\Portfolio\us-election-county-swing\notebooks\..\src\..\data\raw\bls_laus_states.json


## Census demographics

ACS 5-year estimates (2019-2023 vintage) pulled from the Census API. 
Includes population, education, income, race/ethnicity, and employment.

In [7]:
acs_raw = fetch_acs_demographics(API_KEY, vintage=2023)
print(f"Shape: {acs_raw.shape}")
acs_raw.head()

Fetching ACS 2023 5-year data from Census API...
Saved 3222 counties to c:\Users\kaleb\OneDrive\Desktop\Portfolio\us-election-county-swing\notebooks\..\src\..\data\raw\acs_demographics.csv
Shape: (3222, 19)


,NAME,B01003_001E,B01002_001E,B19013_001E,B15003_001E,B15003_022E,B15003_023E,B15003_024E,B15003_025E,B23025_003E,B23025_005E,B03002_001E,B03002_003E,B03002_012E,B02001_002E,B02001_003E,state,county,FIPS
0,"Autauga County, Alabama",59285.0,39.2,69841.0,40767.0,6518.0,4006.0,599.0,407.0,27070.0,688.0,59285.0,42497.0,2188.0,43616.0,11829.0,01,001,01001
1,"Baldwin County, Alabama",239945.0,43.7,75019.0,171988.0,35348.0,14835.0,3843.0,2382.0,113171.0,3615.0,239945.0,195347.0,13393.0,198721.0,19144.0,01,003,01003
2,"Barbour County, Alabama",24757.0,40.7,44290.0,17628.0,1179.0,559.0,193.0,90.0,9074.0,518.0,24757.0,10807.0,1490.0,10891.0,11616.0,01,005,01005
3,"Bibb County, Alabama",22152.0,41.3,51215.0,15931.0,1121.0,537.0,92.0,77.0,9375.0,936.0,22152.0,16333.0,744.0,16634.0,4587.0,01,007,01007
4,"Blount County, Alabama",59292.0,40.9,61096.0,40991.0,4088.0,1843.0,287.0,168.0,27180.0,1586.0,59292.0,50374.0,5962.0,53062.0,747.0,01,009,01009


In [8]:
raw_dir = os.path.join('..', 'data', 'raw')
print("Raw data files:")
for f in sorted(os.listdir(raw_dir)):
    path = os.path.join(raw_dir, f)
    if os.path.isfile(path):
        size_mb = os.path.getsize(path) / 1e6
        print(f"  {f}: {size_mb:.1f} MB")

Raw data files:
  acs_demographics.csv: 0.5 MB
  bls_laus_states.json: 0.1 MB
  countypres_2000-2024.csv: 10.2 MB
  gazetteer_counties.txt: 0.6 MB
  nchs_urban_rural.csv: 0.2 MB
